# Universidad Federico Santa María - 2026
## FINANZAS
### Profesor Fernando Díaz H.

# WACC: Aplicación a una Empresa que No Transa en Bolsa

## El caso: Chick-fil-A

**Chick-fil-A** es una de las cadenas de comida rápida más grandes de Estados Unidos (ventas del sistema cercanas a **USD 24 mil millones** en 2025). Sin embargo, es una empresa **privada**, controlada por la familia Cathy: **sus acciones no se transan en bolsa**.

Supongamos que el directorio evalúa un plan de expansión y necesita una **tasa de descuento** para los flujos de los nuevos locales. El problema es que **no podemos estimar el beta de Chick-fil-A**: no hay serie de precios de su acción.

La solución estándar es el **método de empresas comparables** (*pure play*): usamos empresas que sí transan en bolsa y que tienen un **riesgo operacional (de los activos)** similar.

### ¿Qué haremos?
1. Elegiremos un grupo de **empresas comparables** que transan en bolsa.
2. Descargaremos sus precios y estimaremos su **beta de mercado apalancado** ($\beta^L$) con el modelo de mercado.
3. **Desapalancaremos** cada beta usando su estructura de capital ($B/E$) para obtener el **beta de los activos** ($\beta^U$).
4. Combinaremos los $\beta^U$ de las comparables en un **beta de activos del sector**.
5. **Reapalancaremos** ese beta con la estructura de capital **objetivo** de Chick-fil-A.
6. Calcularemos el **costo del patrimonio** ($r_E$) con el CAPM y el **costo de la deuda** ($r_B$) con un **rating sintético**.
7. Calcularemos el **WACC** y analizaremos su **sensibilidad** al nivel de endeudamiento.

## Recordatorio de fórmulas (ver presentación *Estructura de Capital*)

**Beta apalancado y desapalancado** (Hamada, deuda libre de riesgo, con impuestos):
$$
\beta^L = \beta^U\left[1 + (1-\tau_c)\frac{B}{E}\right]
\qquad\Longleftrightarrow\qquad
\beta^U = \frac{\beta^L}{1 + (1-\tau_c)\,B/E}
$$

**Costo del patrimonio** (CAPM):
$$
r_E = r_f + \beta^L\,\big(\mathbb{E}[r_M] - r_f\big)
$$

**Costo de la deuda**:
$$
r_B = r_f + \text{spread de crédito}
$$

**Costo de capital promedio ponderado**:
$$
WACC = \frac{E}{B+E}\,r_E + \frac{B}{B+E}\,r_B\,(1-\tau_c)
$$

## Cargando las librerías

In [ ]:
# Instala los paquetes que falten (en Colab tidyquant no viene preinstalado; tarda 1-2 minutos)
pkgs <- c("tidyquant", "dplyr", "tidyr", "ggplot2", "scales", "broom", "readr")
nuevos <- pkgs[!pkgs %in% rownames(installed.packages())]
if (length(nuevos) > 0) install.packages(nuevos)

In [ ]:
suppressPackageStartupMessages({
  library(tidyquant)
  library(dplyr)
  library(tidyr)
  library(ggplot2)
  library(scales)
  library(broom)
  library(readr)
})
options(digits = 4, width = 120)

## Parámetros del caso

Todos los **supuestos** del análisis están en esta celda. Modifíquelos para ver cómo cambia el resultado.

| Parámetro | Valor | Comentario |
|---|---|---|
| `TICKERS` | MCD, YUM, QSR, DPZ, WING, WEN | Comparables: cadenas de comida rápida con modelo de franquicias |
| `MERCADO` | ^GSPC | S\&P 500 como proxy del portafolio de mercado |
| `INICIO`, `FIN` | ago-2021 a ago-2026 | 60 retornos mensuales (5 años), práctica estándar |
| `TAU` | 21% | Tasa legal federal de impuesto corporativo en EE.UU. |
| `W_B_OBJ` | 20% | **Supuesto**: estructura de capital objetivo de Chick-fil-A, $B/(B+E)$ |
| `EBIT_V` | 5% | **Supuesto**: EBIT / Valor de la firma (equivale a un múltiplo $V/EBIT \approx 20\times$, similar al de las comparables) |

**¿Por qué estas comparables?** Chick-fil-A es una cadena de comida rápida (*quick service restaurant*) que opera con franquiciados. Las comparables comparten ese negocio:

- **MCD** (McDonald's), **YUM** (KFC, Taco Bell, Pizza Hut), **QSR** (Restaurant Brands: Burger King, Popeyes, Tim Hortons), **DPZ** (Domino's), **WING** (Wingstop, especializada en pollo) y **WEN** (Wendy's).
- Excluimos cadenas *fast casual* con locales propios (por ejemplo Chipotle), cuyo riesgo operacional es distinto.

In [ ]:
EMPRESA  <- "Chick-fil-A"
TICKERS  <- c("MCD", "YUM", "QSR", "DPZ", "WING", "WEN")
NOMBRES  <- c(MCD = "McDonald's", YUM = "Yum! Brands", QSR = "Restaurant Brands Intl.",
              DPZ = "Domino's Pizza", WING = "Wingstop", WEN = "Wendy's")
MERCADO  <- "^GSPC"
INICIO   <- "2021-08-01"
FIN      <- "2026-09-01"

TAU      <- 0.21    # tasa de impuesto corporativo
W_B_OBJ  <- 0.20    # B/(B+E) objetivo de la empresa privada (supuesto)
EBIT_V   <- 0.05    # EBIT / Valor de la firma (supuesto, para el rating sintético)

## Estructura de capital de las comparables

Para desapalancar necesitamos la razón **deuda / patrimonio** de cada comparable, **a valor de mercado**:

- $E$: **capitalización bursátil** (precio × número de acciones).
- $B$: **deuda total**. Usamos el valor libro como aproximación del valor de mercado de la deuda (práctica habitual cuando la deuda no transa).

Los datos corresponden a **Yahoo Finance, *Key Statistics***, al 24-sep-2026 (deuda del último trimestre reportado, junio 2026). Cifras en **miles de millones de USD**.

> **Nota sobre arriendos:** desde la norma ASC 842, la "Total Debt" de Yahoo incluye los **pasivos por arriendo**. En restaurantes estos montos son relevantes. Incluirlos es consistente, siempre que tratemos igual a la empresa objetivo.

Si desea **actualizar los datos a hoy**, reemplace los valores de la celda siguiente con los de Yahoo Finance (pestaña *Statistics*: *Market Cap* y *Total Debt (mrq)*).

In [ ]:
estructura <- tibble(
  Ticker  = TICKERS,
  Empresa = NOMBRES[TICKERS],
  E = c(214.00, 38.38, 25.08, 9.81, 2.67, 1.30),   # capitalización bursátil (USD miles de millones)
  B = c( 54.60, 13.41, 15.65, 5.12, 1.27, 4.07)    # deuda total (USD miles de millones)
) %>%
  mutate(`B/E` = B / E,
         `B/(B+E)` = B / (B + E))
estructura

Observe la enorme dispersión: McDonald's tiene $B/E \approx 0{,}26$, mientras que **Wendy's** tiene $B/E > 3$. Si comparáramos directamente los betas apalancados, estaríamos mezclando **riesgo del negocio** con **riesgo financiero**. Por eso debemos desapalancar.

## Descarga de precios y cálculo de retornos

Con `tidyquant` descargamos precios **mensuales** (ajustados por dividendos y *splits*) de las comparables y del S\&P 500, y calculamos **retornos logarítmicos mensuales**:
$$
r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$

In [ ]:
precios <- tq_get(c(TICKERS, MERCADO), get = "stock.prices",
                  from = INICIO, to = FIN, periodicity = "monthly") %>%
  select(symbol, date, adjusted)

retornos <- precios %>%
  group_by(symbol) %>%
  arrange(date) %>%
  mutate(ret = log(adjusted / lag(adjusted))) %>%
  ungroup() %>%
  select(date, symbol, ret) %>%
  pivot_wider(names_from = symbol, values_from = ret) %>%
  drop_na()

cat(sprintf("Período: %s a %s  (%d meses)\n",
            format(min(retornos$date), "%Y-%m"), format(max(retornos$date), "%Y-%m"), nrow(retornos)))
tail(retornos, 5)

In [ ]:
# Evolución de un dólar invertido en cada acción y en el S&P 500
acum <- retornos %>%
  pivot_longer(-date, names_to = "symbol", values_to = "ret") %>%
  group_by(symbol) %>%
  arrange(date) %>%
  mutate(valor = exp(cumsum(ret)),
         symbol = ifelse(symbol == MERCADO, "S&P 500", symbol)) %>%
  ungroup()

options(repr.plot.width = 10, repr.plot.height = 5)
ggplot(acum, aes(date, valor, color = symbol, linewidth = symbol == "S&P 500")) +
  geom_line() +
  scale_linewidth_manual(values = c(0.6, 1.4), guide = "none") +
  labs(title = "Valor de USD 1 invertido", x = NULL, y = "USD", color = NULL) +
  theme_minimal() + theme(legend.position = "bottom")

## Paso 1: Beta apalancado de cada comparable

Estimamos el **modelo de mercado** por MCO para cada acción $i$:
$$
r_{i,t} = a_i + \beta^L_i\, r_{M,t} + \varepsilon_{i,t}
$$

El $\hat\beta^L_i$ estimado mide el riesgo sistemático del **patrimonio** de cada empresa, que incluye tanto el riesgo del negocio como el riesgo financiero de su deuda.

In [ ]:
betas <- lapply(TICKERS, function(t) {
  modelo <- lm(retornos[[t]] ~ retornos[[MERCADO]])
  coefs  <- tidy(modelo)
  tibble(Ticker = t,
         beta_L = coefs$estimate[2],
         e.e.   = coefs$std.error[2],
         R2     = glance(modelo)$r.squared,
         N      = nobs(modelo))
}) %>% bind_rows()
betas

**Interpretación.** Los restaurantes de comida rápida son en general **defensivos**: sus betas son menores a 1, porque la demanda por comida barata se mantiene relativamente estable en el ciclo económico. Wingstop es la excepción: su beta alto refleja una acción de **crecimiento** muy volátil. Los $R^2$ son bajos (15–20%): la mayor parte del riesgo de cada acción es **idiosincrático** (diversificable).

## Paso 2: Desapalancar

Con la estructura de capital de cada comparable, obtenemos su **beta de activos** (beta desapalancado):
$$
\beta^U_i = \frac{\beta^L_i}{1 + (1-\tau_c)\,(B/E)_i}
$$

El beta de activos mide solo el **riesgo del negocio**, que es lo que Chick-fil-A tiene en común con las comparables.

In [ ]:
comp <- betas %>%
  left_join(estructura %>% select(Ticker, `B/E`), by = "Ticker") %>%
  mutate(beta_U = beta_L / (1 + (1 - TAU) * `B/E`))
comp

In [ ]:
valor_firma <- estructura$E + estructura$B
beta_U_mediana   <- median(comp$beta_U)
beta_U_promedio  <- mean(comp$beta_U)
beta_U_ponderado <- weighted.mean(comp$beta_U, w = valor_firma)

cat(sprintf("Mediana de beta_U                      : %.3f\n", beta_U_mediana))
cat(sprintf("Promedio simple de beta_U              : %.3f\n", beta_U_promedio))
cat(sprintf("Promedio ponderado por valor de firma  : %.3f\n", beta_U_ponderado))

BETA_U <- beta_U_mediana   # usamos la mediana: es robusta a valores extremos

In [ ]:
comp %>%
  select(Ticker, `Beta apalancado` = beta_L, `Beta desapalancado` = beta_U) %>%
  pivot_longer(-Ticker, names_to = "Tipo", values_to = "beta") %>%
  mutate(Ticker = factor(Ticker, levels = TICKERS)) %>%
  ggplot(aes(Ticker, beta, fill = Tipo)) +
  geom_col(position = "dodge") +
  geom_hline(yintercept = BETA_U, linetype = "dashed") +
  annotate("text", x = 0.6, y = BETA_U + 0.06, hjust = 0,
           label = sprintf("Mediana beta_U = %.2f", BETA_U)) +
  scale_fill_manual(values = c("#780404", "#E37D00")) +
  labs(title = "Betas de las comparables: apalancado vs. desapalancado",
       x = NULL, y = "Beta", fill = NULL) +
  theme_minimal() + theme(legend.position = "bottom")

**¿Por qué la mediana?** Con pocas comparables, un solo valor extremo puede mover mucho el promedio. Aquí hay dos:

- **Wingstop** tiene un $\beta^U$ muy alto (acción de crecimiento con alta volatilidad).
- **Wendy's** tiene un $\beta^U$ muy bajo. Con $B/E > 3$, su deuda **no es libre de riesgo**, y la fórmula de Hamada (que supone $\beta_B = 0$) le atribuye todo el riesgo al patrimonio, subestimando el beta de activos. Volveremos a esto en la sección de extensiones.

La mediana es menos sensible a estos dos casos.

## Paso 3: Reapalancar con la estructura de capital de Chick-fil-A

Chick-fil-A es conocida por su financiamiento **conservador**. Suponemos una estructura de capital **objetivo** de $B/(B+E) = 20\%$, es decir:
$$
\frac{B}{E} = \frac{0{,}20}{0{,}80} = 0{,}25
$$

Reapalancamos el beta de activos del sector:
$$
\beta^L_{\text{Chick-fil-A}} = \beta^U\left[1 + (1-\tau_c)\frac{B}{E}\right]
$$

In [ ]:
BE_OBJ     <- W_B_OBJ / (1 - W_B_OBJ)
BETA_L_OBJ <- BETA_U * (1 + (1 - TAU) * BE_OBJ)
cat(sprintf("B/E objetivo           : %.3f\n", BE_OBJ))
cat(sprintf("Beta de activos (β^U)  : %.3f\n", BETA_U))
cat(sprintf("Beta reapalancado (β^L): %.3f\n", BETA_L_OBJ))

## Paso 4: Tasa libre de riesgo y prima por riesgo de mercado

**Tasa libre de riesgo.** Usamos el rendimiento del **bono del Tesoro de EE.UU. a 10 años** (serie `DGS10` de FRED, Banco de la Reserva Federal de St. Louis). El plazo largo es consistente con el horizonte de los proyectos que se evalúan.

**Prima por riesgo de mercado (MRP).** Usamos el **promedio histórico** del factor `Mkt-RF` de Fama-French (sitio de **Kenneth French**), desde 1926. Es el mismo factor que usamos en el notebook de Fama-French.

> **Advertencia:** `Mkt-RF` es el exceso de retorno del mercado sobre las **letras del Tesoro de corto plazo**, por lo que es algo mayor que la prima sobre el bono a 10 años. Una alternativa muy usada en la práctica es la **prima implícita** que publica Aswath Damodaran cada mes; puede ingresarla en `MRP_MANUAL`.

In [ ]:
RF <- tryCatch({
  dgs10 <- tq_get("DGS10", get = "economic.data", from = "2025-01-01") %>% drop_na()
  ultimo <- tail(dgs10, 1)
  cat(sprintf("Bono del Tesoro a 10 años al %s: %.2f%%\n", format(ultimo$date, "%d-%m-%Y"), ultimo$price))
  ultimo$price / 100
}, error = function(e) {
  cat("No se pudo descargar FRED. Se usa r_f = 5,11% (23-sep-2026); actualícelo.\n")
  0.0511
})

In [ ]:
MRP_MANUAL <- NA   # por ejemplo 0.045 para usar la prima implícita de Damodaran

ff_url   <- "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_CSV.zip"
temp_zip <- tempfile(fileext = ".zip")
download.file(ff_url, temp_zip, quiet = TRUE)
ff_csv <- unzip(temp_zip, exdir = tempdir())

ff <- suppressWarnings(read_csv(ff_csv, skip = 3, show_col_types = FALSE, name_repair = "unique_quiet")) %>%
  rename(fecha = 1) %>%
  filter(grepl("^[0-9]{6}$", trimws(fecha))) %>%       # solo filas mensuales (AAAAMM)
  mutate(`Mkt-RF` = as.numeric(`Mkt-RF`))

MRP_HIST <- mean(ff$`Mkt-RF`) * 12 / 100
cat(sprintf("Prima histórica Mkt-RF (%s a %s): %.2f%% anual\n",
            trimws(ff$fecha[1]), trimws(tail(ff$fecha, 1)), 100 * MRP_HIST))

MRP <- ifelse(is.na(MRP_MANUAL), MRP_HIST, MRP_MANUAL)
cat(sprintf("Prima por riesgo de mercado utilizada: %.2f%%\n", 100 * MRP))

## Paso 5: Costo del patrimonio

$$
r_E = r_f + \beta^L_{\text{Chick-fil-A}}\,\text{MRP}
$$

In [ ]:
R_E <- RF + BETA_L_OBJ * MRP
cat(sprintf("r_E = %.2f%% + %.3f x %.2f%% = %.2f%%\n", 100 * RF, BETA_L_OBJ, 100 * MRP, 100 * R_E))

## Paso 6: Costo de la deuda con un rating sintético

Chick-fil-A no tiene bonos que transen en bolsa, así que no observamos directamente su tasa de endeudamiento. Usamos el método del **rating sintético** de Aswath Damodaran:

1. Se calcula la **cobertura de intereses**: $\text{Cobertura} = \dfrac{EBIT}{\text{Gastos por intereses}}$.
2. La cobertura se asocia a un **rating** (AAA, AA, …) según una tabla construida con empresas que sí tienen rating.
3. Cada rating tiene un **spread** sobre la tasa libre de riesgo, observado en bonos transados: $r_B = r_f + \text{spread}$.

La tabla siguiente es la de Damodaran para **grandes empresas no financieras, enero 2026**.

Como no conocemos el EBIT de Chick-fil-A, suponemos $EBIT/V = 5\%$ (parámetro `EBIT_V`). Así:
$$
\text{Cobertura} = \frac{EBIT}{r_B\,B} = \frac{EBIT/V}{r_B \cdot B/V}
$$
Como $r_B$ depende del rating y el rating depende de $r_B$, resolvemos con unas pocas **iteraciones**.

In [ ]:
# Damodaran, "Ratings, Interest Coverage Ratios and Default Spread", enero 2026
tabla_rating <- tribble(
  ~desde, ~hasta, ~Rating,    ~Spread,
  -Inf,   0.20,   "D2/D",     0.1900,
  0.20,   0.65,   "C2/C",     0.1600,
  0.65,   0.80,   "Ca2/CC",   0.1261,
  0.80,   1.25,   "Caa/CCC",  0.0885,
  1.25,   1.50,   "B3/B-",    0.0509,
  1.50,   1.75,   "B2/B",     0.0321,
  1.75,   2.00,   "B1/B+",    0.0275,
  2.00,   2.25,   "Ba2/BB",   0.0184,
  2.25,   2.50,   "Ba1/BB+",  0.0138,
  2.50,   3.00,   "Baa2/BBB", 0.0111,
  3.00,   4.25,   "A3/A-",    0.0089,
  4.25,   5.50,   "A2/A",     0.0078,
  5.50,   6.50,   "A1/A+",    0.0070,
  6.50,   8.50,   "Aa2/AA",   0.0055,
  8.50,   Inf,    "Aaa/AAA",  0.0040
)

rating_sintetico <- function(cobertura) {
  if (is.infinite(cobertura) && cobertura > 0) {        # sin deuda: mejor rating posible
    fila <- tail(tabla_rating, 1)
  } else {
    fila <- tabla_rating %>% filter(desde <= cobertura, cobertura < hasta)
  }
  list(rating = fila$Rating, spread = fila$Spread)
}

costo_deuda <- function(w_B, rf = RF, ebit_v = EBIT_V, iteraciones = 30) {
  if (w_B == 0) {
    r <- rating_sintetico(Inf)
    return(list(rating = r$rating, r_B = rf + r$spread, cobertura = Inf))
  }
  r_B <- rf
  for (i in seq_len(iteraciones)) {
    cobertura <- ebit_v / (r_B * w_B)
    r <- rating_sintetico(cobertura)
    r_B <- rf + r$spread
  }
  list(rating = r$rating, r_B = r_B, cobertura = cobertura)
}

tabla_rating

In [ ]:
deuda     <- costo_deuda(W_B_OBJ)
RATING    <- deuda$rating
R_B       <- deuda$r_B
cat(sprintf("Cobertura de intereses : %.2f veces\n", deuda$cobertura))
cat(sprintf("Rating sintético       : %s\n", RATING))
cat(sprintf("Costo de la deuda r_B  : %.2f%%  (antes de impuestos)\n", 100 * R_B))
cat(sprintf("Costo después de impuestos r_B(1-tau): %.2f%%\n", 100 * R_B * (1 - TAU)))

## Paso 7: El WACC de Chick-fil-A

$$
WACC = \frac{E}{B+E}\,r_E + \frac{B}{B+E}\,r_B\,(1-\tau_c)
$$

In [ ]:
W_E_OBJ <- 1 - W_B_OBJ
WACC    <- W_E_OBJ * R_E + W_B_OBJ * R_B * (1 - TAU)

resumen <- tibble(
  Componente = c("Tasa libre de riesgo (r_f)", "Prima por riesgo de mercado",
                 "Beta de activos (mediana comparables)", "B/E objetivo",
                 "Beta reapalancado", "Costo del patrimonio (r_E)",
                 paste0("Costo de la deuda (r_B, rating ", RATING, ")"), "Tasa de impuestos",
                 "B/(B+E) objetivo", "WACC"),
  Valor = c(RF, MRP, BETA_U, BE_OBJ, BETA_L_OBJ, R_E, R_B, TAU, W_B_OBJ, WACC))
resumen

Este es el costo de capital con el que Chick-fil-A debería descontar los **flujos de caja libres** de un proyecto de **riesgo similar al de su negocio actual** (por ejemplo, abrir nuevos restaurantes en EE.UU.).

Observe que el WACC es **menor** que $r_E$: la deuda es más barata que el patrimonio y, además, sus intereses generan un **escudo fiscal**.

## Sensibilidad: WACC y estructura de capital

¿Qué pasa si Chick-fil-A se endeuda más? Repetimos el cálculo para distintos niveles de $B/(B+E)$:

- Más deuda ⇒ mayor $\beta^L$ ⇒ mayor $r_E$ (Proposición II de M\&M).
- Más deuda ⇒ más **escudo fiscal** ⇒ el WACC tiende a bajar.
- Pero más deuda ⇒ menor cobertura ⇒ **peor rating** ⇒ mayor $r_B$.

El resultado conecta con el **trade-off** entre escudo fiscal y costos de quiebra que vimos en la presentación (enfoque de Leland).

In [ ]:
sens <- lapply(seq(0, 0.60, by = 0.025), function(w_B) {
  be  <- w_B / (1 - w_B)
  b_L <- BETA_U * (1 + (1 - TAU) * be)
  r_E <- RF + b_L * MRP
  d   <- costo_deuda(w_B)
  tibble(`B/(B+E)` = w_B, beta_L = b_L, r_E = r_E, Rating = d$rating, r_B = d$r_B,
         WACC = (1 - w_B) * r_E + w_B * d$r_B * (1 - TAU))
}) %>% bind_rows()

optimo <- sens %>% slice_min(WACC, n = 1, with_ties = FALSE)
cat(sprintf("WACC mínimo: %.2f%% con B/(B+E) = %.1f%% (rating %s)\n",
            100 * optimo$WACC, 100 * optimo$`B/(B+E)`, optimo$Rating))

sens %>% mutate(across(c(`B/(B+E)`, r_E, r_B, WACC), ~ percent(.x, accuracy = 0.01)),
                beta_L = round(beta_L, 3))

In [ ]:
sens %>%
  transmute(w_B = `B/(B+E)`, `r_E` = r_E, `r_B(1-tau)` = r_B * (1 - TAU), WACC = WACC) %>%
  pivot_longer(-w_B, names_to = "Tasa", values_to = "valor") %>%
  mutate(Tasa = factor(Tasa, levels = c("r_E", "r_B(1-tau)", "WACC"))) %>%
  ggplot(aes(w_B, valor, color = Tasa, linewidth = Tasa)) +
  geom_line() +
  geom_vline(xintercept = W_B_OBJ, linetype = "dotted", color = "gray40") +
  geom_point(data = optimo, aes(`B/(B+E)`, WACC), inherit.aes = FALSE, size = 3) +
  scale_color_manual(values = c("#780404", "#1f4e9c", "#E37D00")) +
  scale_linewidth_manual(values = c(0.9, 0.9, 1.6), guide = "none") +
  scale_x_continuous(labels = percent) +
  scale_y_continuous(labels = percent) +
  labs(title = paste0(EMPRESA, ": costo de capital según endeudamiento"),
       subtitle = "Línea punteada: estructura objetivo; punto: WACC mínimo",
       x = "B / (B + E)", y = "Tasa anual", color = NULL) +
  theme_minimal() + theme(legend.position = "bottom")

**Lectura del gráfico.**

- Con poca deuda, el WACC **baja** a medida que aumenta $B/(B+E)$: domina el escudo fiscal.
- Cuando la cobertura de intereses cae, el **rating empeora** y el costo de la deuda salta. A partir de ese punto el WACC **sube**.
- Existe entonces un rango de endeudamiento que **minimiza** el WACC, coherente con la teoría del *trade-off*.

**Limitación.** En este ejercicio $r_E$ se calcula con Hamada, que supone deuda libre de riesgo. Con niveles altos de deuda, parte del riesgo lo asumen los acreedores y la fórmula deja de ser precisa.

## Extensión: ¿y si la deuda de las comparables es riesgosa?

Cuando la deuda es riesgosa ($\beta_B > 0$), la relación entre betas es (ver presentación):
$$
\beta^L = \beta^U + (\beta^U - \beta_B)(1-\tau_c)\frac{B}{E}
\qquad\Longrightarrow\qquad
\beta^U = \frac{\beta^L + \beta_B(1-\tau_c)\,B/E}{1 + (1-\tau_c)\,B/E}
$$

Un valor razonable para $\beta_B$ se obtiene del CAPM aplicado a la deuda: si el spread es de 1–2% y la prima de mercado es cercana a 8%, entonces $\beta_B \approx \text{spread}/\text{MRP} \approx 0{,}1\text{–}0{,}25$.

Veamos cómo cambian los betas de activos con $\beta_B = 0{,}25$ para todas las comparables.

In [ ]:
BETA_B <- 0.25
comp <- comp %>%
  mutate(`beta_U (B riesgosa)` = (beta_L + BETA_B * (1 - TAU) * `B/E`) / (1 + (1 - TAU) * `B/E`))
print(comp %>% select(Ticker, beta_L, `B/E`, beta_U, `beta_U (B riesgosa)`))
cat(sprintf("Mediana beta_U con deuda libre de riesgo: %.3f\n", median(comp$beta_U)))
cat(sprintf("Mediana beta_U con deuda riesgosa (beta_B = %.2f): %.3f\n",
            BETA_B, median(comp$`beta_U (B riesgosa)`)))

El ajuste es más importante para las empresas **más endeudadas**, especialmente Wendy's. Con deuda riesgosa, su beta de activos se vuelve mucho más parecido al del resto del sector.

## Exportar tablas a LaTeX

Las siguientes celdas generan las tablas para la presentación en Overleaf (`Comparables_WACC.tex` y `WACC_Resumen.tex`).

In [ ]:
num <- function(x, d = 3) gsub(".", "{,}", formatC(x, format = "f", digits = d), fixed = TRUE)
pct <- function(x) paste0(gsub(".", "{,}", formatC(100 * x, format = "f", digits = 2), fixed = TRUE), "\\%")

tabla_latex <- function(filas, encabezado, alineacion, titulo, extra = NULL) {
  c("\\begin{table}[!htbp] \\centering \\scriptsize",
    paste0("\\caption{", titulo, "}"),
    paste0("\\begin{tabular}{", alineacion, "}"),
    "\\toprule",
    paste0(paste(encabezado, collapse = " & "), " \\\\"),
    "\\midrule",
    sapply(filas, function(f) paste0(paste(f, collapse = " & "), " \\\\")),
    extra,
    "\\bottomrule", "\\end{tabular}", "\\end{table}")
}

filas_comp <- lapply(seq_len(nrow(comp)), function(i) {
  with(comp[i, ], c(Ticker, NOMBRES[Ticker], num(beta_L), paste0("(", num(e.e.), ")"),
                    num(R2), num(`B/E`), num(beta_U)))
})
periodo <- sprintf("%s a %s", format(min(retornos$date), "%Y-%m"), format(max(retornos$date), "%Y-%m"))
tex_comp <- tabla_latex(
  filas_comp,
  c("Ticker", "Empresa", "$\\hat\\beta^L$", "e.e.", "$R^2$", "$B/E$", "$\\beta^U$"),
  "llccccc", paste0("Betas de las comparables (", periodo, ")"),
  extra = c("\\midrule", paste0("\\multicolumn{6}{l}{\\textbf{Mediana}} & \\textbf{", num(BETA_U), "} \\\\")))

filas_res <- list(
  c("Tasa libre de riesgo, $r_f$", pct(RF)),
  c("Prima por riesgo de mercado", pct(MRP)),
  c("Beta de activos, $\\beta^U$", num(BETA_U)),
  c("$B/(B+E)$ objetivo", pct(W_B_OBJ)),
  c("Beta reapalancado, $\\beta^L$", num(BETA_L_OBJ)),
  c("Costo del patrimonio, $r_E$", pct(R_E)),
  c(paste0("Costo de la deuda, $r_B$ (rating ", RATING, ")"), pct(R_B)),
  c("Tasa de impuestos, $\\tau_c$", pct(TAU)),
  c("\\textbf{WACC}", paste0("\\textbf{", pct(WACC), "}")))
tex_res <- tabla_latex(filas_res, c("Componente", "Valor"), "lr", paste0("WACC de ", EMPRESA))

writeLines(tex_comp, "Comparables_WACC.tex")
writeLines(tex_res, "WACC_Resumen.tex")
cat(tex_comp, sep = "\n"); cat("\n")
cat(tex_res, sep = "\n")
cat("\nTablas exportadas: Comparables_WACC.tex, WACC_Resumen.tex\n")

## Discusión: ¿qué tan confiable es este WACC?

1. **Elección de comparables.** Es la decisión más importante. Chick-fil-A opera solo en EE.UU., casi sin locales propios y con crecimiento sostenido; ninguna comparable es idéntica.
2. **Error de estimación.** Los errores estándar de los betas son grandes (0,13 a 0,44). Promediar varias comparables reduce ese error.
3. **Estructura de capital objetivo.** El 20% es un supuesto. Lo relevante es la estructura **objetivo de largo plazo**, no la de un año en particular.
4. **Prima por riesgo de mercado.** Pasar de la prima histórica a la implícita puede mover el WACC en más de un punto porcentual. Pruebe con `MRP_MANUAL`.
5. **Empresa privada.** El CAPM supone que el dueño está **diversificado**. Los dueños de una empresa familiar suelen no estarlo, y además sus acciones son **ilíquidas**. En la práctica, muchos analistas agregan una **prima por iliquidez** o usan un *beta total*.

### Preguntas para el alumno
- ¿Cómo cambia el WACC si excluye a Wingstop y Wendy's de las comparables?
- ¿Qué pasa si usa la prima implícita de Damodaran en vez de la histórica?
- Si Chick-fil-A quisiera financiar su expansión con 40% de deuda, ¿qué rating obtendría y cuál sería su WACC?